In [1]:
import pandas as pd, numpy as np, sqlite3, lightgbm as lgb
import warnings; warnings.filterwarnings("ignore")

con = sqlite3.connect("forecast.db")
# (paste the FEATURE_SQL query string from 03_database Cell 2 here)

FEATURE_SQL = """
WITH daily AS (
    SELECT s.item_id, s.store_id, s.date, s.units,
           c.event_name_1, c.snap_CA, c.wm_yr_wk
    FROM sales s
    JOIN calendar c ON s.date = c.date
)
SELECT d.item_id, d.store_id, d.date, d.units,
       CAST(strftime('%w', d.date) AS INT)            AS dayofweek,
       CAST(strftime('%m', d.date) AS INT)            AS month,
       CASE WHEN d.event_name_1 IS NOT NULL THEN 1 ELSE 0 END AS is_event,
       d.snap_CA,
       LAG(d.units, 7)  OVER w                        AS lag_7,
       LAG(d.units, 14) OVER w                        AS lag_14,
       LAG(d.units, 28) OVER w                        AS lag_28,
       AVG(d.units) OVER (w ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)  AS roll_mean_7,
       AVG(d.units) OVER (w ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_mean_28,
       p.sell_price
FROM daily d
LEFT JOIN prices p
  ON p.item_id = d.item_id AND p.store_id = d.store_id AND p.wm_yr_wk = d.wm_yr_wk
WINDOW w AS (PARTITION BY d.item_id, d.store_id ORDER BY d.date)
ORDER BY d.item_id, d.store_id, d.date
"""

feat_sql = pd.read_sql(FEATURE_SQL, con, parse_dates=["date"])

df = feat_sql.dropna(subset=["lag_7","lag_14","lag_28","roll_mean_7","roll_mean_28","sell_price"])
df = df.copy()
g = df.groupby(["item_id","store_id"])["sell_price"]
df["price_lag_7"] = g.shift(7)
df["price_change"] = (df["sell_price"] - df["price_lag_7"]) / df["price_lag_7"]
df["price_rel"] = df["sell_price"] / g.transform("mean")
df = df.dropna(subset=["price_change"]).copy()
df["category"] = df["item_id"].str.split("_").str[0]
df["is_weekend"] = df["dayofweek"].isin([0,6]).astype(int)
for c in ["item_id","store_id","category"]:
    df[c] = df[c].astype("category")

FEATURES3 = ["dayofweek","is_weekend","month","is_event","snap_CA",
             "lag_7","lag_14","lag_28","roll_mean_7","roll_mean_28",
             "sell_price","price_change","price_rel","store_id","category","item_id"]

cutoff3 = df["date"].max() - pd.Timedelta(28, unit="D")
train3, test3 = df[df["date"] <= cutoff3], df[df["date"] > cutoff3].copy()

def eval_mae(preds):
    t = test3.assign(pred=np.clip(preds, 0, None))
    return t.groupby(["item_id","store_id"], observed=True).apply(
        lambda x: np.mean(np.abs(x["units"] - x["pred"]))).mean()

print(f"Baseline to beat (q50): 3.94")

Baseline to beat (q50): 3.94


In [2]:
for p in (1.1, 1.2, 1.3, 1.5):
    m = lgb.LGBMRegressor(objective="tweedie", tweedie_variance_power=p,
                          n_estimators=800, learning_rate=0.05,
                          num_leaves=63, random_state=42)
    m.fit(train3[FEATURES3], train3["units"],
          categorical_feature=["store_id","category","item_id"])
    print(f"tweedie p={p}:  MAE {eval_mae(m.predict(test3[FEATURES3])):.3f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014369 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1924
[LightGBM] [Info] Number of data points in the train set: 526524, number of used features: 16
[LightGBM] [Info] Start training from score 2.600066
tweedie p=1.1:  MAE 4.021
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018884 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1924
[LightGBM] [Info] Number of data points in the train set: 526524, number of used features: 16
[LightGBM] [Info] Start training from score 2.600066
tweedie p=1.2:  MAE 4.004
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019515 seconds.
You can set `force_row_wise=tr

In [3]:
!python.exe -m pip install optuna

In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

val_cut = cutoff3 - pd.Timedelta(28, unit="D")
tr_in  = train3[train3["date"] <= val_cut]
val_in = train3[train3["date"] >  val_cut]

def objective(trial):
    params = dict(
        objective="tweedie",
        tweedie_variance_power=trial.suggest_float("tvp", 1.05, 1.6),
        n_estimators=trial.suggest_int("n_estimators", 400, 1500),
        learning_rate=trial.suggest_float("lr", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        min_child_samples=trial.suggest_int("min_child", 10, 100),
        feature_fraction=trial.suggest_float("ff", 0.6, 1.0),
        lambda_l2=trial.suggest_float("l2", 0.0, 10.0),
        random_state=42, verbosity=-1)
    m = lgb.LGBMRegressor(**params)
    m.fit(tr_in[FEATURES3], tr_in["units"],
          categorical_feature=["store_id","category","item_id"])
    v = val_in.assign(pred=np.clip(m.predict(val_in[FEATURES3]), 0, None))
    return v.groupby(["item_id","store_id"], observed=True).apply(
        lambda x: np.mean(np.abs(x["units"] - x["pred"]))).mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40, show_progress_bar=True)
print("best val MAE:", round(study.best_value, 3))
print("best params:", study.best_params)

  0%|          | 0/40 [00:00<?, ?it/s]

best val MAE: 4.224
best params: {'tvp': 1.4527380467716335, 'n_estimators': 1433, 'lr': 0.06016547602262111, 'num_leaves': 156, 'min_child': 70, 'ff': 0.7871119781670656, 'l2': 0.9920439495810274}


In [5]:
bp = study.best_params
final = lgb.LGBMRegressor(objective="tweedie",
    tweedie_variance_power=bp["tvp"], n_estimators=bp["n_estimators"],
    learning_rate=bp["lr"], num_leaves=bp["num_leaves"],
    min_child_samples=bp["min_child"], feature_fraction=bp["ff"],
    lambda_l2=bp["l2"], random_state=42, verbosity=-1)
final.fit(train3[FEATURES3], train3["units"],
          categorical_feature=["store_id","category","item_id"])
print(f"TUNED TWEEDIE test MAE: {eval_mae(final.predict(test3[FEATURES3])):.3f}  (champion: 3.94)")

TUNED TWEEDIE test MAE: 3.982  (champion: 3.94)


In [6]:
vol = train3.groupby(["item_id","store_id"], observed=True)["units"].mean()
slow = vol[vol < 3].index
t = test3.assign(pred=np.clip(final.predict(test3[FEATURES3]), 0, None))
t["key"] = list(zip(t["item_id"], t["store_id"]))
slow_mae = t[t["key"].isin(slow)].groupby("key").apply(
    lambda x: np.mean(np.abs(x["units"] - x["pred"]))).mean()
print(f"Slow movers (<3/day avg): {len(slow)} series, tuned MAE {slow_mae:.3f}")
print("Compare mean prediction on slow movers:",
      round(t[t["key"].isin(slow)]["pred"].mean(), 2), "vs actual",
      round(t[t["key"].isin(slow)]["units"].mean(), 2))

Slow movers (<3/day avg): 2 series, tuned MAE 2.388
Compare mean prediction on slow movers: 3.14 vs actual 3.52
